In [ ]:
/**
 https://www.bambooweekly.com/government-corruption/ 
 https://www.bambooweekly.com/government-corruption-436/ 
 https://github.com/JoergEm/Bamboo-Weekly/tree/main
*/

In [ ]:
// Imports
import java.net.{HttpURLConnection, URL}
import java.nio.file.{Files, Paths, StandardCopyOption}
import java.net.URI
import java.net.http.{HttpClient, HttpRequest, HttpResponse}
import $ivy.`org.apache.poi:poi-ooxml:5.4.1`
import org.apache.poi.xssf.usermodel.XSSFWorkbook
import scala.jdk.CollectionConverters._
import java.io.FileInputStream

In [ ]:
// Function creating local folders
def createFolders(folders: List[String]): Boolean = {
  try {
    folders.foreach { folder =>
      val folderPath = Paths.get(System.getProperty("user.dir"), folder)

      if (!Files.exists(folderPath)) {
        Files.createDirectories(folderPath)
      }
    }

    println("Folders ✅")
    true
  } catch {
    case ex: Exception =>
      println(s"Error ❌ ${ex.getMessage}")
      false
  }
}

In [ ]:
// Function downloading data locally
def downloadData(url: String, filename: String): Boolean = {
  try {
    val client = HttpClient.newBuilder()
      .followRedirects(HttpClient.Redirect.NORMAL)
      .build()

    val request = HttpRequest.newBuilder()
      .uri(URI.create(url))
      .GET()
      .build()

    val response = client.send(
      request,
      HttpResponse.BodyHandlers.ofByteArray()
    )

    if (response.statusCode() == 200) {
      Files.write(Paths.get(filename), response.body())
      true
    } else {
      println(s"Error ❌ ${response.statusCode()}")
      false
    }
  } catch {
    case ex: Exception =>
      println(s"Error ❌ ${ex.getMessage}")
      false
  }
}

In [ ]:
// Functions to read Excel into CPI Object
case class CPI(rank: Int, country: String, score: Double)

def readExcel(filepath: java.nio.file.Path): List[List[String]] = {
  val file = new FileInputStream(filepath.toFile)

  try {
    val workbook = new XSSFWorkbook(file)

    try {
      val sheet = workbook.getSheet("CPI 2022 (final)")

      sheet.iterator().asScala
        .drop(2) // skiprows = 2
        .map { row =>
          row.cellIterator().asScala
            .map(_.toString)
            .toList
        }
        .toList

    } finally {
      workbook.close()
    }
  } finally {
    file.close()
  }
}

def readCPI(filepath: java.nio.file.Path): List[CPI] = {
  val rows = readExcel(filepath)

  rows.tail.map { row =>
    CPI(
      rank = row(4).toDouble.toInt,
      country = row(0),
      score = row(3).toDouble
    )
  }
}

In [ ]:
// Links and folders to recieve data and read into DataFrame
val url = "https://images.transparencycdn.org/images/CPI2022_GlobalResultsTrends.xlsx"
val filename = "CPI2022_GlobalResultsTrends.xlsx"
val folders = List("data", "results")
val filepath = Paths.get(folders.head, filename)

createFolders(folders)

val data: List[CPI] = {
  if (!Files.exists(filepath)) {
    if (!downloadData(url, filepath.toString)) {
      println("Error ❌")
      List.empty[CPI]
    } else {
        println("Data ✅")
        println(s"File: ${filepath.toAbsolutePath}")
        readCPI(filepath)      
    }
  } else {
      println("Data loaded from existing file ✅")
      println(s"File: ${filepath.toAbsolutePath}")
      readCPI(filepath)
  }
}

In [ ]:
// According to Transparency International, what five countries were least corrupt in 2022?
def top5(data: List[CPI]): Unit = {
  data.sortBy(_.rank).take(5).foreach { c =>
    println(f"${c.rank}%3d  ${c.country}%-20s ${c.score}%5.1f")
  }
}

top5(data)

In [ ]:
// According to the same data, what five countries were most corrupt in 20…
def least5(data: List[CPI]): Unit = {
  data.sortBy(_.rank).takeRight(5).foreach { c =>
    println(f"${c.rank}%3d  ${c.country}%-20s ${c.score}%5.1f")
  }
}

least5(data)